# 1_channel_statistics.ipynb

## 透明转发架构下 LEO 卫星下行链路：NTN 信道统计验证

本 notebook 基于原 `channel_response.ipynb` 的信道生成方式整理，目标不是做多用户 MIMO 条件数分析，而是验证单用户 LEO 下行链路中的 NTN 信道统计特性。

输出三类图：

1. 三种场景的 RMS delay spread CDF；
2. 三种场景的平均接收信道增益 / 接收功率统计；
3. 三种场景的 OFDM 频域响应幅度样例。

注意：这里不再使用 condition number CDF，因为当前毕设中期主线是单用户下行链路，不是多用户 MIMO 条件数分析。

In [ ]:
# Cell 1：导入库与基础环境设置

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

gpu_num = 0
os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)

from sionna.phy.channel import subcarrier_frequencies, cir_to_ofdm_channel
from sionna.phy.channel.tr38811 import AntennaArray, DenseUrban, Urban, SubUrban
from sionna.phy.channel.tr38811.utils import gen_single_sector_topology as gen_ntn_topology

tf.get_logger().setLevel("ERROR")
print("TensorFlow version:", tf.__version__)

### Cell 1 作用

导入 TensorFlow、NumPy、Matplotlib，以及 Sionna/OpenNTN 中的 TR 38.811 NTN 信道模型。  
这里使用的 `gen_single_sector_topology` 与原 `channel_response.ipynb` 保持一致。

In [ ]:
# Cell 2：统一仿真参数

SCENARIOS = {
    "dur": "Dense Urban",
    "urb": "Urban",
    "sur": "SubUrban",
}

direction = "downlink"
num_ut = 1
num_ut_ant = 1
num_bs_ant = 1

satellite_height = 600000.0
elevation_angle = 50.0

carrier_frequency = 3.5e9
fft_size = 52 * 12
subcarrier_spacing = 15e3
num_ofdm_symbols = 12
sampling_frequency = fft_size * subcarrier_spacing

batch_size = 512

enable_pathloss = True
enable_shadow_fading = True
doppler_enabled = False

np.random.seed(1)
tf.random.set_seed(1)

print("direction =", direction)
print("num_ut =", num_ut)
print("num_ut_ant =", num_ut_ant)
print("num_bs_ant =", num_bs_ant)
print("carrier_frequency =", carrier_frequency)
print("satellite_height =", satellite_height)

### Cell 2 作用

统一设置参数，保证后面三张图都使用同一套单用户下行链路配置。  
这里明确使用 `direction = "downlink"`、`num_ut = 1`、`num_ut_ant = 1`、`num_bs_ant = 1`，避免把上行、多用户、MIMO 条件数分析混入当前中期主线。

In [ ]:
# Cell 3：场景模型创建函数

def build_channel_model(scenario,
                        carrier_frequency=carrier_frequency,
                        elevation_angle=elevation_angle,
                        direction=direction,
                        num_ut_ant=num_ut_ant,
                        num_bs_ant=num_bs_ant,
                        enable_pathloss=enable_pathloss,
                        enable_shadow_fading=enable_shadow_fading,
                        doppler_enabled=doppler_enabled):
    """
    根据场景字符串创建 3GPP TR 38.811 NTN 信道模型。

    scenario:
        "dur" -> DenseUrban
        "urb" -> Urban
        "sur" -> SubUrban
    """
    ut_array = AntennaArray(
        num_rows=1,
        num_cols=num_ut_ant,
        polarization="single",
        polarization_type="V",
        antenna_pattern="omni",
        carrier_frequency=carrier_frequency,
    )

    bs_array = AntennaArray(
        num_rows=1,
        num_cols=num_bs_ant,
        polarization="single",
        polarization_type="V",
        antenna_pattern="aperture",
        carrier_frequency=carrier_frequency,
    )

    cls_map = {
        "dur": DenseUrban,
        "urb": Urban,
        "sur": SubUrban,
    }

    if scenario not in cls_map:
        raise ValueError(f"Unknown scenario: {scenario}. Use one of {list(cls_map.keys())}")

    channel_model = cls_map[scenario](
        carrier_frequency=carrier_frequency,
        ut_array=ut_array,
        bs_array=bs_array,
        direction=direction,
        elevation_angle=elevation_angle,
        enable_pathloss=enable_pathloss,
        enable_shadow_fading=enable_shadow_fading,
        doppler_enabled=doppler_enabled,
    )

    return channel_model

### Cell 3 作用

把 DenseUrban、Urban、SubUrban 三种信道模型封装成统一函数。  
注意这里没有手动设置 K 因子，因为当前公开 notebook 接口里没有直接暴露 `K_factor` 参数。

In [ ]:
# Cell 4：CIR 采样函数

def sample_cir(scenario,
               batch_size=batch_size,
               num_time_steps=1,
               sampling_frequency=sampling_frequency,
               bs_height=satellite_height,
               elevation_angle=elevation_angle,
               enable_pathloss=enable_pathloss,
               enable_shadow_fading=enable_shadow_fading,
               doppler_enabled=doppler_enabled):
    """
    生成指定场景下的 topology，并采样 CIR。

    返回：
        path_coefficients:
            通常为 [batch, num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths, num_time_steps]
        path_delays:
            通常为 [batch, num_rx, num_tx, num_paths]
    """
    channel_model = build_channel_model(
        scenario=scenario,
        elevation_angle=elevation_angle,
        enable_pathloss=enable_pathloss,
        enable_shadow_fading=enable_shadow_fading,
        doppler_enabled=doppler_enabled,
    )

    topology = gen_ntn_topology(
        batch_size=batch_size,
        num_ut=num_ut,
        scenario=scenario,
        bs_height=bs_height,
        elevation_angle=elevation_angle,
    )

    channel_model.set_topology(*topology)

    path_coefficients, path_delays = channel_model(num_time_steps, sampling_frequency)

    return path_coefficients, path_delays

### Cell 4 作用

沿用原 notebook 的核心逻辑：

```python
topology = gen_ntn_topology(...)
channel_model.set_topology(*topology)
```

区别是这里固定为单用户下行链路，并封装成函数，方便后面三种场景循环调用。

In [ ]:
# Cell 5：RMS delay spread 计算函数

def compute_rms_delay_spread(path_coefficients, path_delays, eps=1e-30):
    """
    根据路径系数和路径时延计算每个 batch 样本的 RMS delay spread。

    mean_delay = sum(P_i * tau_i) / sum(P_i)
    rms_delay_spread = sqrt(sum(P_i * (tau_i - mean_delay)^2) / sum(P_i))

    其中 P_i 由路径系数幅度平方得到。
    """
    a = path_coefficients.numpy()
    tau = path_delays.numpy()

    power = np.mean(np.abs(a) ** 2, axis=-1)
    power = np.mean(power, axis=(1, 2, 3, 4))

    tau = np.mean(tau, axis=(1, 2))

    power_sum = np.sum(power, axis=-1, keepdims=True) + eps
    mean_delay = np.sum(power * tau, axis=-1, keepdims=True) / power_sum
    rms_delay = np.sqrt(np.sum(power * (tau - mean_delay) ** 2, axis=-1) / np.squeeze(power_sum, axis=-1))

    return rms_delay

### Cell 5 作用

实现 RMS delay spread 公式。  
这里明确不是 condition number CDF，而是基于每条路径时延和路径功率计算的多径时延扩展统计。

In [ ]:
# Cell 6：图 1 —— 三种场景 RMS delay spread CDF

rms_ds_results = {}

for sc in SCENARIOS:
    print(f"Sampling CIR for scenario: {sc} ({SCENARIOS[sc]})")
    a, tau = sample_cir(
        scenario=sc,
        batch_size=batch_size,
        num_time_steps=1,
        enable_pathloss=True,
        enable_shadow_fading=True,
        doppler_enabled=False,
    )
    rms_ds = compute_rms_delay_spread(a, tau)
    rms_ds_results[sc] = rms_ds

plt.figure(figsize=(7, 5))

for sc, rms_ds in rms_ds_results.items():
    x = np.sort(rms_ds * 1e9)
    y = np.arange(1, len(x) + 1) / len(x)
    plt.plot(x, y, label=SCENARIOS[sc])

plt.xlabel("RMS delay spread (ns)")
plt.ylabel("CDF")
plt.title("CDF of RMS Delay Spread for NTN Scenarios")
plt.grid(True, which="both")
plt.legend()
plt.tight_layout()
plt.show()

### 图 1 物理意义

RMS delay spread 描述多径分量在时延上的分散程度。  
数值越大，说明多径越分散，频率选择性越强，对 OFDM 信道估计和均衡越不利。

一般解释趋势：

- Dense Urban：建筑遮挡和散射更强，RMS delay spread 通常更大；
- Urban：中等；
- SubUrban：散射较弱，LOS 更稳定，RMS delay spread 通常更小。

由于 NTN 模型含随机拓扑和大尺度参数，单次仿真曲线可能有交叉；答辩时重点解释整体统计趋势。

In [ ]:
# Cell 7：图 2 —— 接收信道增益 / 平均接收功率代理指标统计

def compute_average_channel_gain_db(path_coefficients, eps=1e-30):
    """
    用 CIR 路径功率和作为平均接收信道增益的代理指标。

    注意：
    - 如果 enable_pathloss=True，则该指标包含路径损耗和阴影衰落影响；
    - 这里不是直接读取 path loss，而是用 sum(|a_i|^2) 表示等效接收信道增益；
    - dB 值越小，表示接收信道增益越低，也可理解为链路越差。
    """
    a = path_coefficients.numpy()

    power = np.mean(np.abs(a) ** 2, axis=-1)
    gain = np.sum(power, axis=-1)
    gain = np.mean(gain, axis=(1, 2, 3, 4))

    gain_db = 10 * np.log10(gain + eps)
    return gain_db


gain_results = {}

for sc in SCENARIOS:
    print(f"Sampling channel gain for scenario: {sc} ({SCENARIOS[sc]})")
    a, tau = sample_cir(
        scenario=sc,
        batch_size=batch_size,
        num_time_steps=1,
        enable_pathloss=True,
        enable_shadow_fading=True,
        doppler_enabled=False,
    )
    gain_results[sc] = compute_average_channel_gain_db(a)

plt.figure(figsize=(7, 5))
data = [gain_results[sc] for sc in SCENARIOS]
labels = [SCENARIOS[sc] for sc in SCENARIOS]

plt.boxplot(data, labels=labels, showmeans=True)
plt.ylabel("Average channel gain proxy (dB)")
plt.title("Average Received Channel Gain Statistics")
plt.grid(True, axis="y")
plt.tight_layout()
plt.show()

for sc in SCENARIOS:
    print(f"{SCENARIOS[sc]:12s}: mean = {np.mean(gain_results[sc]):.2f} dB, std = {np.std(gain_results[sc]):.2f} dB")

### 图 2 物理意义

这里没有强行读取底层 path loss 参数，而是使用 `sum(|a_i|^2)` 作为平均接收信道增益代理指标。

解释方式：

- 该值越低，说明等效接收功率越低，链路越差；
- Dense Urban 通常遮挡更强、阴影衰落更明显，因此平均接收信道增益应更低；
- SubUrban 通常 LOS 条件更好，因此平均接收信道增益应更高。

如果运行结果出现少量交叉，优先看均值和箱线图整体位置，不要只看单个样本。

In [ ]:
# Cell 8：图 3 —— 三种场景 OFDM 频域响应幅度样例

frequencies = subcarrier_frequencies(fft_size, subcarrier_spacing)

plt.figure(figsize=(8, 5))

for sc in SCENARIOS:
    a, tau = sample_cir(
        scenario=sc,
        batch_size=1,
        num_time_steps=1,
        enable_pathloss=False,
        enable_shadow_fading=False,
        doppler_enabled=False,
    )

    h = cir_to_ofdm_channel(frequencies, a, tau, normalize=True)
    h_np = h.numpy()

    h_vec = np.squeeze(h_np)
    if h_vec.ndim > 1:
        h_vec = h_vec.reshape(-1, h_vec.shape[-1])[0]

    mag = np.abs(h_vec)
    mag = mag / (np.mean(mag) + 1e-30)

    plt.plot(np.arange(fft_size), mag, label=SCENARIOS[sc])

plt.xlabel("Subcarrier index")
plt.ylabel("Normalized channel magnitude |H(f)|")
plt.title("Example OFDM Frequency Response Magnitude")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

### 图 3 物理意义

OFDM 频域响应幅度 `|H(f)|` 展示不同子载波上的信道增益变化。

解释方式：

- 如果 `|H(f)|` 随子载波变化明显，说明信道具有频率选择性；
- RMS delay spread 越大，通常频域起伏越明显；
- 该图是样例图，主要用于说明 NTN 多径信道经过 OFDM 后会形成不同子载波上的不同复信道系数。

In [ ]:
# Cell 9：可选检查 —— 打印 topology 返回内容的类型和 shape

# 这个 cell 用来回答“为什么不直接改速度控制多普勒”的工程问题：
# 先看 gen_ntn_topology 到底返回了哪些张量、shape 是什么。
# 不建议在中期主线里直接修改这些张量；这个 cell 只用于检查和答辩解释。

topology = gen_ntn_topology(
    batch_size=2,
    num_ut=num_ut,
    scenario="dur",
    bs_height=satellite_height,
    elevation_angle=elevation_angle,
)

print("Number of topology elements:", len(topology))
for i, item in enumerate(topology):
    if hasattr(item, "shape"):
        print(f"topology[{i}]: type={type(item)}, shape={item.shape}, dtype={getattr(item, 'dtype', None)}")
    else:
        print(f"topology[{i}]: type={type(item)}, value={item}")

### Cell 9 作用

这个 cell 不是主结果，只是用于检查 `gen_ntn_topology(...)` 返回内容。  
如果后续想研究“能否直接改速度”，必须先确认 topology 中哪一项是速度、单位是什么、shape 是什么，以及它是否代表 UT 速度还是卫星相对速度。中期阶段不建议手动改这些张量。